In [1]:
!pip install -q tensorflow pandas scikit-learn


^C
ERROR: Operation cancelled by user


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from collections import defaultdict

HISTORY = 64
FEATURES_PER_BRANCH = 12
CSV_PATH = "combined_gather.csv"
WORK_DIR = os.getcwd()
print("Working directory:", WORK_DIR)


2025-04-29 17:36:14.694539: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-29 17:36:14.694927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-29 17:36:14.743872: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-29 17:36:14.892592: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Working directory: /workspace/transformer_branch_predictor


In [3]:
def encode_type(bt):
    return [1,0,0] if bt == 0 else [0,1,0] if bt == 1 else [0,0,1] if bt == 2 else [0,0,0]

def top4(values):
    cnt = defaultdict(int)
    top = []
    for v in values:
        cnt[v] += 1
        if v not in top:
            if len(top) < 4:
                top.append(v)
            else:
                worst = min(top, key=lambda x: (cnt[x], values.index(x)))
                if cnt[v] > cnt[worst] or (cnt[v] == cnt[worst] and values.index(v) > values.index(worst)):
                    top.remove(worst)
                    top.append(v)
    return top

def onehot(val, four):
    bits = [0]*4
    if val in four:
        bits[four.index(val)] = 1
    return bits


In [4]:
def row_to_sample(row):
    pcs = [row[f'pc{i}'] for i in range(HISTORY)]
    targets = [row[f'target{i}'] for i in range(HISTORY)]
    top_pc = top4(pcs)
    top_targ = top4(targets)
    mat = []
    for i in range(HISTORY):
        mat.append(
            [row[f'taken{i}']] +
            onehot(pcs[i], top_pc) +
            onehot(targets[i], top_targ) +
            encode_type(row[f'type{i}'])
        )
    return np.array(mat, dtype=np.uint8), np.uint8(row['new'])


In [5]:
def load(csv_path, test_split=0.2, batch=128):
    df = pd.read_csv(csv_path)
    X, y = zip(*(row_to_sample(r) for _, r in df.iterrows()))
    X = np.array(X, dtype=np.uint8)
    y = np.array(y, dtype=np.uint8)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=42, stratify=y)
    def tfcast(x, y):
        return tf.cast(x, tf.float32), tf.cast(y, tf.float32)
    train_ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(10000).batch(batch).map(tfcast).prefetch(tf.data.AUTOTUNE)
    test_ds  = tf.data.Dataset.from_tensor_slices((Xte, yte)).batch(batch).map(tfcast).prefetch(tf.data.AUTOTUNE)
    return train_ds, test_ds

train_ds, test_ds = load(CSV_PATH, batch=4096)


2025-04-29 17:39:10.858990: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-29 17:39:10.916262: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-29 17:39:10.916302: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-29 17:39:10.920002: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-29 17:39:10.920036: I external/local_xla/xla/stream_executor

In [10]:
from tensorflow.keras.layers import Input, Dense, Add, LayerNormalization, GlobalAveragePooling1D, MultiHeadAttention
from tensorflow.keras.models import Model

# single layer, 1 head
def build_tiny_transformer():
    inp = Input(shape=(HISTORY, FEATURES_PER_BRANCH), dtype=tf.float32, name="bits")
    x = Dense(8, use_bias=False, name="proj")(inp)
    att = MultiHeadAttention(num_heads=1, key_dim=8, name="mha")(x, x)
    x = Add()([x, att])
    x = LayerNormalization()(x)
    y = Dense(32, activation='relu', name="ff1")(x)
    y = Dense(8, name="ff2")(y)
    x = Add()([x, y])
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation='sigmoid', name="taken_next")(x)
    model = Model(inputs=inp, outputs=out, name="tiny_branch_transformer")
    return model

#single layer, 2 heads
def build_transformer_h2():
    inp = Input((HISTORY, FEATURES_PER_BRANCH), name="bits")
    x = Dense(8, use_bias=False, name="proj_h2")(inp)
    att = MultiHeadAttention(num_heads=2, key_dim=4, name="mha_h2")(x, x)
    x = Add()([x, att]); x = LayerNormalization()(x)
    y = Dense(32, activation="relu", name="ff1_h2")(x)
    y = Dense(8, name="ff2_h2")(y)
    x = Add()([x, y]); x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid", name="taken_next_h2")(x)
    return Model(inp, out, name="transformer_h2")


#single layer, 4 heads
def build_transformer_h4():
    inp = Input((HISTORY, FEATURES_PER_BRANCH), name="bits")
    x = Dense(16, use_bias=False, name="proj_h4")(inp)
    att = MultiHeadAttention(num_heads=4, key_dim=4, name="mha_h4")(x, x)
    x = Add()([x, att]); x = LayerNormalization()(x)
    y = Dense(64, activation="relu", name="ff1_h4")(x)
    y = Dense(16, name="ff2_h4")(y)
    x = Add()([x, y]); x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid", name="taken_next_h4")(x)
    return Model(inp, out, name="transformer_h4")


#two layer, 1 head
def build_transformer_layer2():
    inp = Input((HISTORY, FEATURES_PER_BRANCH), name="bits")
    x = Dense(8, use_bias=False, name="proj_l2")(inp)
    # Block 1
    att1 = MultiHeadAttention(num_heads=1, key_dim=8, name="mha1_l2")(x, x)
    x = LayerNormalization()(Add()([x, att1]))
    y1 = Dense(32, activation="relu", name="ff1_1_l2")(x)
    y1 = Dense(8, name="ff2_1_l2")(y1)
    x = LayerNormalization()(Add()([x, y1]))
    # Block 2
    att2 = MultiHeadAttention(num_heads=1, key_dim=8, name="mha2_l2")(x, x)
    x = LayerNormalization()(Add()([x, att2]))
    y2 = Dense(32, activation="relu", name="ff1_2_l2")(x)
    y2 = Dense(8, name="ff2_2_l2")(y2)
    x = LayerNormalization()(Add()([x, y2]))
    # Head
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid", name="taken_next_l2")(x)
    return Model(inp, out, name="transformer_layer2")


#two layer, 2 heads
def build_transformer_layer2_h2():
    inp = Input((HISTORY, FEATURES_PER_BRANCH), name="bits")
    x = Dense(16, use_bias=False, name="proj_l2h2")(inp)
    #block 1
    att1 = MultiHeadAttention(num_heads=2, key_dim=8, name="mha1_l2h2")(x, x)
    x = LayerNormalization()(Add()([x, att1]))
    y1 = Dense(64, activation="relu", name="ff1_1_l2h2")(x)
    y1 = Dense(16, name="ff2_1_l2h2")(y1)
    x = LayerNormalization()(Add()([x, y1]))
    #block 2
    att2 = MultiHeadAttention(num_heads=2, key_dim=8, name="mha2_l2h2")(x, x)
    x = LayerNormalization()(Add()([x, att2]))
    y2 = Dense(64, activation="relu", name="ff1_2_l2h2")(x)
    y2 = Dense(16, name="ff2_2_l2h2")(y2)
    x = LayerNormalization()(Add()([x, y2]))
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid", name="taken_next_l2h2")(x)
    return Model(inp, out, name="transformer_layer2_h2")


#three layer, 2 heads
def build_transformer_layer3_h2():
    inp = Input((HISTORY, FEATURES_PER_BRANCH), name="bits")
    x = Dense(16, use_bias=False, name="proj_l3h2")(inp)
    #block 1
    att1 = MultiHeadAttention(num_heads=2, key_dim=8, name="mha1_l3h2")(x, x)
    x = LayerNormalization()(Add()([x, att1]))
    y1 = Dense(64, activation="relu", name="ff1_1_l3h2")(x)
    y1 = Dense(16, name="ff2_1_l3h2")(y1)
    x = LayerNormalization()(Add()([x, y1]))
    #block 2
    att2 = MultiHeadAttention(num_heads=2, key_dim=8, name="mha2_l3h2")(x, x)
    x = LayerNormalization()(Add()([x, att2]))
    y2 = Dense(64, activation="relu", name="ff1_2_l3h2")(x)
    y2 = Dense(16, name="ff2_2_l3h2")(y2)
    x = LayerNormalization()(Add()([x, y2]))
    #block 3
    att3 = MultiHeadAttention(num_heads=2, key_dim=8, name="mha3_l3h2")(x, x)
    x = LayerNormalization()(Add()([x, att3]))
    y3 = Dense(64, activation="relu", name="ff1_3_l3h2")(x)
    y3 = Dense(16, name="ff2_3_l3h2")(y3)
    x = LayerNormalization()(Add()([x, y3]))
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid", name="taken_next_l3h2")(x)
    return Model(inp, out, name="transformer_layer3_h2")


#select model type here
model = build_transformer_layer3_h2()
model.summary()


Model: "transformer_layer3_h2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 bits (InputLayer)           [(None, 64, 12)]             0         []                            
                                                                                                  
 proj_l3h2 (Dense)           (None, 64, 16)               192       ['bits[0][0]']                
                                                                                                  
 mha1_l3h2 (MultiHeadAttent  (None, 64, 16)               1088      ['proj_l3h2[0][0]',           
 ion)                                                                'proj_l3h2[0][0]']           
                                                                                                  
 add_2 (Add)                 (None, 64, 16)               0         ['proj_l3h

In [7]:
model.summary()
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=30,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )
]

model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=250,
    callbacks=callbacks
)



Model: "tiny_branch_transformer"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 bits (InputLayer)           [(None, 64, 12)]             0         []                            
                                                                                                  
 proj (Dense)                (None, 64, 8)                96        ['bits[0][0]']                
                                                                                                  
 mha (MultiHeadAttention)    (None, 64, 8)                288       ['proj[0][0]',                
                                                                     'proj[0][0]']                
                                                                                                  
 add (Add)                   (None, 64, 8)                0         ['proj[0

2025-04-29 17:39:13.380913: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:467] Loaded cuDNN version 90000
2025-04-29 17:39:13.958068: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f888ed1a4c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-04-29 17:39:13.958112: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Laptop GPU, Compute Capability 8.6
2025-04-29 17:39:13.971120: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1745948354.078379   77494 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


36/36 [==============================] - 6s 72ms/step - loss: 0.6689 - accuracy: 0.6015 - val_loss: 0.6154 - val_accuracy: 0.6633 - lr: 0.0010
Epoch 2/250
36/36 [==============================] - 3s 88ms/step - loss: 0.5887 - accuracy: 0.6882 - val_loss: 0.5593 - val_accuracy: 0.7039 - lr: 0.0010
Epoch 3/250
36/36 [==============================] - 3s 89ms/step - loss: 0.5385 - accuracy: 0.7180 - val_loss: 0.5295 - val_accuracy: 0.7172 - lr: 0.0010
Epoch 4/250
36/36 [==============================] - 3s 90ms/step - loss: 0.5235 - accuracy: 0.7206 - val_loss: 0.5236 - val_accuracy: 0.7144 - lr: 0.0010
Epoch 5/250
36/36 [==============================] - 3s 88ms/step - loss: 0.5187 - accuracy: 0.7189 - val_loss: 0.5193 - val_accuracy: 0.7183 - lr: 0.0010
Epoch 6/250
36/36 [==============================] - 3s 88ms/step - loss: 0.5147 - accuracy: 0.7235 - val_loss: 0.5152 - val_accuracy: 0.7198 - lr: 0.0010
Epoch 7/250
36/36 [==============================] - 3s 88ms/step - loss: 0.5107 -

In [9]:
#save as Keras H5
h5_path = os.path.join(WORK_DIR, "branch_predictor.h5")
model.save(h5_path)
print("Saved Keras H5 model to:", h5_path)

print(model.input_names)

#also save savedmodel format
saved_model_dir = os.path.join(WORK_DIR, "branch_predictor_savedmodel")
tf.saved_model.save(model, saved_model_dir)
print("Saved SavedModel to:", saved_model_dir)


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Saved Keras H5 model to: /workspace/transformer_branch_predictor/branch_predictor.h5
['bits']
INFO:tensorflow:Assets written to: /workspace/transformer_branch_predictor/branch_predictor_savedmodel/assets


INFO:tensorflow:Assets written to: /workspace/transformer_branch_predictor/branch_predictor_savedmodel/assets


Saved SavedModel to: /workspace/transformer_branch_predictor/branch_predictor_savedmodel
